In [7]:
import os
import gc
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from datetime import datetime
from itertools import repeat
from multiprocessing import Pool, cpu_count
from filesplit.split import Split

matplotlib.use('Agg')

BASE_DIR = os.path.dirname(os.path.abspath('__file__'))
DATA_DIR = os.path.join(os.path.dirname(BASE_DIR), 'data')

In [8]:
def sizeof_fmt(num, suffix="B"):
    for unit in ("", "Ki", "Mi", "Gi", "Ti", "Pi", "Ei", "Zi"):
        if abs(num) < 1024.0:
            return f"{num:3.1f}{unit}{suffix}"
        num /= 1024.0
    return f"{num:.1f}Yi{suffix}"

def count_time(func):
    def wrapper(*args, **kwargs):
        start = datetime.now()
        result = func(*args, **kwargs)
        elapsed = datetime.now() - start
        print(f"Czas wykonania {func.__name__}: {elapsed}")
        return result
    return wrapper

In [9]:
def apply_args_and_kwargs(func, args, kwargs):
    return func(*args, **kwargs)

def starmap_with_kwargs(pool, func, args_iter, kwargs_iter):
    args_for_starmap = zip(repeat(func), args_iter, kwargs_iter)
    return pool.starmap(apply_args_and_kwargs, args_for_starmap)

def split_file(filepath, chunksize, destination):
    os.makedirs(destination, exist_ok=True)
    split = Split(filepath, destination)
    split.bylinecount(linecount=chunksize, includeheader=True)

def load_files_mp(directory, n_processes):
    files = [[f"{directory}/{f}"] for f in sorted(os.listdir(directory)) if f.endswith(".csv")]
    kwargs_list = [{'on_bad_lines': "skip"} for _ in range(len(files))]
    pool = Pool(processes=n_processes)
    results = starmap_with_kwargs(pool, pd.read_csv, files, kwargs_list)
    pool.close()
    pool.join()
    return pd.concat(results, ignore_index=True)

def sum_likes_from_file(filepath):
    df = pd.read_csv(filepath, on_bad_lines='skip')
    return df['likes'].sum()

### Zadanie 1

In [10]:
df1 = pd.read_parquet(os.path.join(DATA_DIR, '0000.parquet'))
df2 = pd.read_parquet(os.path.join(DATA_DIR, '0001.parquet'))
df = pd.concat([df1, df2], ignore_index=True)
del df1, df2

In [11]:
print(df.dtypes)

sid                    int64
sid_profile            int64
post_id                  str
profile_id             int64
date                     str
post_type              int64
description              str
likes                  int64
comments               int64
username                 str
bio                      str
following              int64
followers              int64
num_posts              int64
is_business_account     bool
lang                     str
category                 str
dtype: object


In [12]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2294716 entries, 0 to 2294715
Data columns (total 17 columns):
 #   Column               Dtype
---  ------               -----
 0   sid                  int64
 1   sid_profile          int64
 2   post_id              str  
 3   profile_id           int64
 4   date                 str  
 5   post_type            int64
 6   description          str  
 7   likes                int64
 8   comments             int64
 9   username             str  
 10  bio                  str  
 11  following            int64
 12  followers            int64
 13  num_posts            int64
 14  is_business_account  bool 
 15  lang                 str  
 16  category             str  
dtypes: bool(1), int64(9), str(7)
memory usage: 1.0 GB


In [13]:
pd.options.display.float_format = '{:.2f}'.format
df.describe()

,sid,sid_profile,profile_id,post_type,likes,comments,following,followers,num_posts
count,2294716.00,2294716.00,2294716.00,2294716.00,2294716.00,2294716.00,2294716.00,2294716.00,2294716.00
mean,20104487.78,2675541.83,2243935718.36,1.02,705.86,12.47,1482.61,43840.06,1296.60
std,10684810.93,1156602.73,2984035219.40,0.15,9540.98,202.36,4188.41,1165456.35,2713.65
min,6025.00,-1.00,4.00,1.00,0.00,0.00,0.00,0.00,0.00
25%,10601129.00,2106827.00,183263166.00,1.00,23.00,0.00,458.00,440.00,175.00
50%,18830599.50,2982047.00,620321076.00,1.00,55.00,2.00,893.00,1049.00,528.00
75%,28713018.75,3469724.00,3448978951.00,1.00,156.00,7.00,1728.00,3298.00,1376.00
max,43330528.00,4510785.00,14188428836.00,3.00,2331497.00,173988.00,1190947.00,285457645.00,125220.00


In [ ]:
mem_shallow = sum(df.memory_usage())
mem_deep = sum(df.memory_usage(deep=True))
print(f"Pamięć RAM (shallow): {sizeof_fmt(mem_shallow)}")
print(f"Pamięć RAM (deep):    {sizeof_fmt(mem_deep)}")

In [ ]:
for col in df.columns:
    print(f"{col:25s}: {sizeof_fmt(df[col].memory_usage(deep=True)):>10s}  ({df[col].dtype})")

### Zadanie 2

In [ ]:
df_orig = df.copy()
df_opt = pd.DataFrame()

for col in ['sid', 'sid_profile', 'profile_id']:
    min_val, max_val = df_orig[col].min(), df_orig[col].max()
    print(f"{col}: min={min_val}, max={max_val}")
    if min_val >= 0 and max_val <= np.iinfo(np.int32).max:
        df_opt[col] = df_orig[col].astype(np.int32)
    else:
        df_opt[col] = df_orig[col]

df_opt['post_id'] = df_orig['post_id'].astype('category')
df_opt['date'] = pd.to_datetime(df_orig['date'], errors='coerce')
df_opt['post_type'] = pd.to_numeric(df_orig['post_type'], downcast='integer')
df_opt['description'] = df_orig['description']
df_opt['likes'] = pd.to_numeric(df_orig['likes'], downcast='integer')
df_opt['comments'] = pd.to_numeric(df_orig['comments'], downcast='integer')
df_opt['username'] = df_orig['username'].astype('category')
df_opt['bio'] = df_orig['bio']
df_opt['following'] = pd.to_numeric(df_orig['following'], downcast='integer')
df_opt['followers'] = pd.to_numeric(df_orig['followers'], downcast='integer')
df_opt['num_posts'] = pd.to_numeric(df_orig['num_posts'], downcast='integer')
df_opt['is_business_account'] = df_orig['is_business_account']
df_opt['lang'] = df_orig['lang'].astype('category')
df_opt['category'] = df_orig['category'].astype('category')

In [ ]:
mem_orig = sum(df_orig.memory_usage(deep=True))
mem_opt = sum(df_opt.memory_usage(deep=True))

print(f"Pamięć oryginalna:      {sizeof_fmt(mem_orig)}")
print(f"Pamięć zoptymalizowana: {sizeof_fmt(mem_opt)}")
print(f"Redukcja: {(1 - mem_opt / mem_orig) * 100:.1f}%")

In [ ]:
for col in df_opt.columns:
    print(f"{col:25s}: {sizeof_fmt(df_opt[col].memory_usage(deep=True)):>10s}  ({df_opt[col].dtype})")

In [ ]:
labels = ['Oryginalna', 'Zoptymalizowana']
values = [mem_orig / (1024 ** 2), mem_opt / (1024 ** 2)]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(labels, values, color=['#e74c3c', '#2ecc71'], width=0.5, edgecolor='black')

for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 10,
            f'{val:.1f} MiB', ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_ylabel('Pamięć RAM (MiB)', fontsize=12)
ax.set_title('Porównanie zużycia pamięci RAM\n(oryginalne vs zoptymalizowane typy danych)', fontsize=13)
ax.set_ylim(0, max(values) * 1.15)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'zadanie2_porownanie_pamieci.png'), dpi=150)
plt.show()

### Zadanie 3

In [ ]:
results = {}

start = datetime.now()
_ = df_orig.groupby('username').agg({'likes': 'mean'})
t1_orig = datetime.now() - start

start = datetime.now()
_ = df_opt.groupby('username').agg({'likes': 'mean'})
t1_opt = datetime.now() - start

print(f"Oryginalna:      {t1_orig}")
print(f"Zoptymalizowana: {t1_opt}")
results['Grupowanie +\nśrednia likes'] = (t1_orig.total_seconds(), t1_opt.total_seconds())

In [ ]:
start = datetime.now()
_ = df_orig[df_orig['likes'] > 100]
t2_orig = datetime.now() - start

start = datetime.now()
_ = df_opt[df_opt['likes'] > 100]
t2_opt = datetime.now() - start

print(f"Oryginalna:      {t2_orig}")
print(f"Zoptymalizowana: {t2_opt}")
results['Filtrowanie\nlikes > 100'] = (t2_orig.total_seconds(), t2_opt.total_seconds())

In [ ]:
start = datetime.now()
_ = df_orig.sort_values('followers', ascending=False)
t3_orig = datetime.now() - start

start = datetime.now()
_ = df_opt.sort_values('followers', ascending=False)
t3_opt = datetime.now() - start

print(f"Oryginalna:      {t3_orig}")
print(f"Zoptymalizowana: {t3_opt}")
results['Sortowanie\npo followers'] = (t3_orig.total_seconds(), t3_opt.total_seconds())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(results))
width = 0.35
orig_times = [v[0] for v in results.values()]
opt_times = [v[1] for v in results.values()]

bars1 = ax.bar(x - width/2, orig_times, width, label='Oryginalna', color='#e74c3c')
bars2 = ax.bar(x + width/2, opt_times, width, label='Zoptymalizowana', color='#2ecc71')

ax.set_ylabel('Czas (sekundy)')
ax.set_title('Porównanie czasów operacji: oryginalne vs zoptymalizowane typy')
ax.set_xticks(x)
ax.set_xticklabels(results.keys())
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'zadanie3_porownanie_czasow.png'), dpi=150)
plt.show()

### Zadanie 4

In [ ]:
csv_path = os.path.join(BASE_DIR, 'instagram_data.csv')
df_orig.to_csv(csv_path, header=True, index=False)
csv_size = os.path.getsize(csv_path)
print(f"Rozmiar pliku CSV: {sizeof_fmt(csv_size)}")

In [ ]:
parquet_files = [os.path.join(DATA_DIR, '0000.parquet'), os.path.join(DATA_DIR, '0001.parquet')]
parquet_total = sum(os.path.getsize(f) for f in parquet_files)
print(f"Suma rozmiarów plików parquet: {sizeof_fmt(parquet_total)}")
print(f"CSV jest {csv_size / parquet_total:.1f}x większy niż parquet")

In [ ]:
del df_orig, df_opt, df
gc.collect()

### Zadanie 5

In [ ]:
n_cores = cpu_count()
print(f"Liczba rdzeni CPU: {n_cores}")

In [ ]:
@count_time
def read_csv_whole():
    return pd.read_csv(csv_path, header=0)

df_csv1 = read_csv_whole()
del df_csv1

In [ ]:
@count_time
def read_csv_chunked(chunksize):
    chunks = pd.read_csv(csv_path, header=0, chunksize=chunksize)
    return pd.concat(chunks, ignore_index=True)

for cs in [500_000, 1_000_000, 2_000_000]:
    print(f"chunksize={cs:,}")
    df_csv2 = read_csv_chunked(cs)
    del df_csv2

In [ ]:
split_dir = os.path.join(BASE_DIR, 'data_split')
if os.path.exists(split_dir):
    for f in os.listdir(split_dir):
        os.remove(os.path.join(split_dir, f))
split_file(csv_path, 500_000, split_dir)
n_files = len([f for f in os.listdir(split_dir) if f.endswith('.csv')])
print(f"Podzielono plik na {n_files} części")

In [ ]:
procs_1 = max(1, n_cores - 2)
procs_2 = max(1, (n_cores - 2) * 2)

start = datetime.now()
df_mp1 = load_files_mp(split_dir, procs_1)
t_mp1 = datetime.now() - start
print(f"Multiprocessing ({procs_1} procesów): {t_mp1}")
del df_mp1

try:
    start = datetime.now()
    df_mp2 = load_files_mp(split_dir, procs_2)
    t_mp2 = datetime.now() - start
    print(f"Multiprocessing ({procs_2} procesów): {t_mp2}")
    del df_mp2
except Exception as e:
    print(f"Błąd przy {procs_2} procesach: {type(e).__name__}: {e}")

### Zadanie 6

In [ ]:
csv_files = sorted([
    os.path.join(split_dir, f) for f in os.listdir(split_dir)
    if f.endswith('.csv')
])
print(f"Liczba plików: {len(csv_files)}")

In [ ]:
start = datetime.now()
total_likes_seq = 0
for fpath in csv_files:
    df_chunk = pd.read_csv(fpath, on_bad_lines='skip')
    total_likes_seq += df_chunk['likes'].sum()
t_seq = datetime.now() - start
print(f"Suma likes (sekwencyjnie): {total_likes_seq:,}")
print(f"Czas: {t_seq}")

In [ ]:
n_procs = max(1, cpu_count() - 2)
start = datetime.now()
with Pool(processes=n_procs) as pool:
    partial_sums = pool.map(sum_likes_from_file, csv_files)
total_likes_par = sum(partial_sums)
t_par = datetime.now() - start
print(f"Suma likes (równolegle): {total_likes_par:,}")
print(f"Czas: {t_par}")
print(f"Przyspieszenie: {t_seq.total_seconds() / t_par.total_seconds():.2f}x")